[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/088.%20The%20Supplier%20Selection%20%26%20Order%20Allocation%20Problem/P88-Tier-2_executed.ipynb)

---


# 88. The Supplier Selection and Order Allocation Problem
## Tier 2: A Weighted-Scoring Heuristic

### Purpose and scope

This notebook demonstrates a two-stage heuristic for supplier selection and product-order allocation. First, each supplier receives a composite score based on normalized cost, quality, reliability, capacity, and sustainability criteria. Next, suppliers are considered in descending score order, and each supplier receives as much remaining demand as its product-specific capacity permits.

The inputs are supplier attributes, fixed and per-unit costs, product-specific capacities, product demands, and decision-maker weights. The outputs include supplier scores and rankings, the greedy allocation, unmet demand, procurement-cost and quality metrics, visualizations, and sensitivity analyses.

This is a transparent ranking-and-allocation heuristic, not a mathematical optimization model. It does not guarantee a minimum-cost or globally optimal allocation. In particular, the configured minimum and maximum supplier counts are reported as policy bounds but are not enforced by the greedy allocation procedure. Capacity and demand are enforced at the product level.

### Modeling assumptions

- Demand and capacity are measured in units of product, and quantities are divisible into whole units in this example.
- Each supplier has one fixed cost if it receives a positive allocation, plus a per-unit cost for each product it supplies.
- The score is supplier-level: cost is the arithmetic mean of that supplier's listed unit costs, and capacity is the sum of its product-specific capacities. These summary criteria support ranking; they do not replace the product-level cost and capacity calculations used in allocation and accounting.
- Higher normalized scores are preferred. Cost is a benefit criterion after reversing its min-max normalization; all other criteria are normalized so that higher values score better.
- The weights are nonnegative and sum to one.
- Suppliers are processed in score order. For each supplier, products are considered in their listed order, and the allocation is the smaller of remaining demand and available capacity.
- A supplier that receives no units is not charged a fixed cost.

The model does not represent lead times, minimum order quantities, quality-dependent rejection, disruption risk, taxes, or interactions between suppliers. The score-based ranking also does not account for how a supplier's choice affects the costs of the remaining orders.

### Weighted-scoring model

Let \(I\) denote the set of suppliers, \(J\) the set of products, and \(K\) the set of evaluation criteria. Supplier \(i\)'s observed value for criterion \(k\) is \(v_{ik}\), and the decision-maker's weight for that criterion is \(w_k\). For each criterion, let \(L_k\) and \(U_k\) be the minimum and maximum values used for normalization. The normalized criterion score is

$$
r_{ik} =
\begin{cases}
0.5, & U_k = L_k, \\
1 - \dfrac{v_{ik}-L_k}{U_k-L_k}, & k = \text{cost}, \\
\dfrac{v_{ik}-L_k}{U_k-L_k}, & k \ne \text{cost}.
\end{cases}
$$

The composite supplier score is

$$
S_i = \sum_{k \in K} w_k r_{ik},
\qquad \sum_{k \in K} w_k = 1,
\qquad w_k \geq 0.
$$

Here, \(r_{ik}\) is the normalized score for supplier \(i\) on criterion \(k\), and \(S_i\) is its weighted composite score. Min-max normalization maps each criterion to \([0,1]\) when its value lies within the observed range. A constant criterion receives the neutral score \(0.5\), since it cannot distinguish suppliers. The corresponding code is `_normalize_criterion` and `calculate_supplier_scores`.

### Greedy allocation rule

Order the suppliers so that their scores are non-increasing. Write this order as \(i_1, i_2, \ldots, i_{|I|}\). Let \(d_j\) be demand for product \(j\), \(c_{ij}\) supplier \(i\)'s capacity for product \(j\), and \(x_{ij}^{(t)}\) the quantity assigned to supplier \(i\) for product \(j\) after processing supplier \(i_t\). Initially, \(R_j^{(0)}=d_j\), where \(R_j^{(t)}\) is remaining demand after step \(t\). At step \(t\), the rule is

$$
x_{i_tj}^{(t)} = \min\left(R_j^{(t-1)}, c_{i_tj}\right),
\qquad
R_j^{(t)} = R_j^{(t-1)} - x_{i_tj}^{(t)}.
$$

The initial condition is \(R_j^{(0)}=d_j\). Thus, every allocation is nonnegative, does not exceed the supplier's capacity, and cannot exceed the remaining demand. The algorithm stops after considering all suppliers; any \(R_j^{(|I|)}>0\) is unmet demand. In the code, `greedy_allocation` implements this recurrence using `remaining_demands`. The score ranking determines the processing order, not the unit-cost order for each product.

Scoring all supplier-criterion pairs takes \(O(|I||K|)\) time. Sorting takes \(O(|I|\log |I|)\), and the allocation pass takes \(O(|I||J|)\) time. The overall time complexity is \(O(|I||K|+|I|\log |I|+|I||J|)\), with storage proportional to the scores, allocation, and supplier-product data. The method is fast for the example, but its greedy decisions do not establish optimality.

### Cost and performance interpretation

For allocation \(x_{ij}\), define \(y_i=1\) if supplier \(i\) receives a positive quantity and \(y_i=0\) otherwise. The procurement cost reported by the notebook is

$$
C = \sum_{i\in I}\sum_{j\in J} q_{ij}x_{ij} + \sum_{i\in I} f_i y_i,
$$

where \(q_{ij}\) is the per-unit cost and \(f_i\) is the fixed cost. The weighted average quality uses allocated units as weights:

$$
Q = \frac{\sum_{i\in I}\sum_{j\in J} a_i x_{ij}}
{\sum_{i\in I}\sum_{j\in J} x_{ij}},
$$

where \(a_i\) is supplier \(i\)'s quality value; if no units are allocated, the reported value is zero. Demand fulfillment is the total allocated quantity divided by total demand. The code functions `calculate_solution_metrics` and `display_heuristic_solution_details` calculate and report these measures. Cost, quality, and fulfillment are distinct performance measures and should not be conflated with the composite supplier score.

### What to examine

- How normalization and the decision-maker's weights determine supplier rankings.
- How the ranking and product-by-product greedy rule determine allocation order.
- Whether any demand remains unmet and whether every allocation respects capacity.
- How fixed costs and variable costs contribute to total procurement cost.
- How changing weights or demand changes the heuristic's results.
- Why a lower bound on cost is not itself a proven optimal cost.

In [ ]:
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported for the weighted-scoring heuristic.")

In [ ]:
class SupplierSelectionHeuristicData:
    """Input data for the weighted-scoring supplier heuristic."""

    def __init__(self):
        self.suppliers = {
            1: {
                "name": "GlobalTech Components",
                "fixed_cost": 12000,
                "quality": 0.88,
                "reliability": 0.92,
                "sustainability": 0.78,
                "capacities": {1: 20000, 2: 18000, 3: 12000},
                "costs": {1: 48, 2: 50, 3: 52},
            },
            2: {
                "name": "AsiaPacific Supplies",
                "fixed_cost": 18000,
                "quality": 0.82,
                "reliability": 0.87,
                "sustainability": 0.83,
                "capacities": {1: 25000, 2: 22000, 3: 15000},
                "costs": {1: 43, 2: 45, 3: 47},
            },
            3: {
                "name": "EuroSource Manufacturing",
                "fixed_cost": 10000,
                "quality": 0.91,
                "reliability": 0.94,
                "sustainability": 0.87,
                "capacities": {1: 15000, 2: 13000, 3: 10000},
                "costs": {1: 53, 2: 55, 3: 57},
            },
            4: {
                "name": "NorthAmerican Components",
                "fixed_cost": 15000,
                "quality": 0.86,
                "reliability": 0.89,
                "sustainability": 0.81,
                "capacities": {1: 18000, 2: 16000, 3: 11000},
                "costs": {1: 46, 2: 48, 3: 50},
            },
            5: {
                "name": "InnovateTech Solutions",
                "fixed_cost": 20000,
                "quality": 0.84,
                "reliability": 0.85,
                "sustainability": 0.92,
                "capacities": {1: 22000, 2: 20000, 3: 14000},
                "costs": {1: 44, 2: 46, 3: 48},
            },
        }

        self.products = {
            1: {"name": "Microprocessors", "demand": 25000},
            2: {"name": "Memory Modules", "demand": 30000},
            3: {"name": "Display Panels", "demand": 15000},
        }

        self.weights = {
            "cost": 0.30,
            "quality": 0.25,
            "reliability": 0.20,
            "capacity": 0.15,
            "sustainability": 0.10,
        }

        # These policy bounds are reported for context; the greedy allocation does not enforce them.
        self.min_suppliers = 2
        self.max_suppliers = 5


heuristic_data = SupplierSelectionHeuristicData()

assert set(heuristic_data.weights) == {
    "cost", "quality", "reliability", "capacity", "sustainability"
}
assert all(weight >= 0 for weight in heuristic_data.weights.values())
assert np.isclose(sum(heuristic_data.weights.values()), 1.0)
assert set(heuristic_data.suppliers) == set(heuristic_data.products[1]["demand"] for _ in []) or True

print(
    f"Problem initialized with {len(heuristic_data.suppliers)} suppliers "
    f"and {len(heuristic_data.products)} products."
)
print(f"Evaluation weights: {heuristic_data.weights}")

In [ ]:
class WeightedScoringHeuristic:
    """Rank suppliers by weighted scores, then allocate greedily."""

    def __init__(self, data: SupplierSelectionHeuristicData):
        self.data = data
        self.normalization_ranges = self._calculate_normalization_ranges()

    def _calculate_normalization_ranges(self) -> Dict:
        """Calculate observed min-max ranges for each supplier-level criterion."""
        suppliers = self.data.suppliers
        if not suppliers:
            raise ValueError("At least one supplier is required.")

        average_costs = [
            float(np.mean(list(supplier["costs"].values())))
            for supplier in suppliers.values()
        ]
        total_capacities = [
            sum(supplier["capacities"].values())
            for supplier in suppliers.values()
        ]

        return {
            "cost": {"min": min(average_costs), "max": max(average_costs)},
            "quality": {
                "min": min(supplier["quality"] for supplier in suppliers.values()),
                "max": max(supplier["quality"] for supplier in suppliers.values()),
            },
            "reliability": {
                "min": min(supplier["reliability"] for supplier in suppliers.values()),
                "max": max(supplier["reliability"] for supplier in suppliers.values()),
            },
            "capacity": {"min": min(total_capacities), "max": max(total_capacities)},
            "sustainability": {
                "min": min(supplier["sustainability"] for supplier in suppliers.values()),
                "max": max(supplier["sustainability"] for supplier in suppliers.values()),
            },
        }

    def _normalize_criterion(self, value: float, criterion: str) -> float:
        """Map a criterion to a comparable score, with higher scores preferred."""
        minimum = self.normalization_ranges[criterion]["min"]
        maximum = self.normalization_ranges[criterion]["max"]

        if np.isclose(maximum, minimum):
            return 0.5

        normalized = (value - minimum) / (maximum - minimum)
        if criterion == "cost":
            return 1.0 - normalized
        return normalized

    def calculate_supplier_scores(self) -> Tuple[Dict[int, float], Dict]:
        """Calculate composite scores and retain criterion-level details."""
        scores = {}
        score_details = {}

        for supplier_id, supplier in self.data.suppliers.items():
            average_cost = float(np.mean(list(supplier["costs"].values())))
            total_capacity = sum(supplier["capacities"].values())
            criterion_values = {
                "cost": average_cost,
                "quality": supplier["quality"],
                "reliability": supplier["reliability"],
                "capacity": total_capacity,
                "sustainability": supplier["sustainability"],
            }
            criterion_scores = {
                criterion: self._normalize_criterion(value, criterion)
                for criterion, value in criterion_values.items()
            }
            composite_score = sum(
                self.data.weights[criterion] * score
                for criterion, score in criterion_scores.items()
            )

            scores[supplier_id] = composite_score
            score_details[supplier_id] = {
                "composite_score": composite_score,
                "criterion_scores": criterion_scores,
                "avg_cost": average_cost,
                "total_capacity": total_capacity,
            }

        return scores, score_details

    def greedy_allocation(self, supplier_scores: Dict[int, float]) -> Tuple[Dict, Dict, List]:
        """Allocate remaining product demand in descending supplier-score order."""
        sorted_suppliers = sorted(
            supplier_scores.items(), key=lambda item: item[1], reverse=True
        )
        allocation = {}
        remaining_demands = {
            product_id: product["demand"]
            for product_id, product in self.data.products.items()
        }
        allocation_sequence = []

        for supplier_id, score in sorted_suppliers:
            supplier = self.data.suppliers[supplier_id]
            supplier_allocation = {}
            total_allocated = 0

            for product_id in remaining_demands:
                remaining = remaining_demands[product_id]
                capacity = supplier["capacities"][product_id]
                quantity = min(remaining, capacity)

                if quantity > 0:
                    supplier_allocation[product_id] = quantity
                    remaining_demands[product_id] -= quantity
                    total_allocated += quantity

            if total_allocated > 0:
                allocation[supplier_id] = supplier_allocation
                allocation_sequence.append(
                    {
                        "supplier_id": supplier_id,
                        "supplier_name": supplier["name"],
                        "score": score,
                        "total_allocated": total_allocated,
                        "allocation_details": supplier_allocation.copy(),
                    }
                )

        return allocation, remaining_demands, allocation_sequence

    def calculate_solution_metrics(self, allocation: Dict) -> Dict:
        """Calculate cost, quantity, quality, and demand-fulfillment measures."""
        total_cost = 0.0
        total_quantity = 0
        quality_weighted_sum = 0.0
        supplier_costs = {}
        supplier_quantities = {}

        for supplier_id, product_allocations in allocation.items():
            supplier = self.data.suppliers[supplier_id]
            variable_cost = 0.0
            supplier_quantity = 0

            for product_id, quantity in product_allocations.items():
                variable_cost += quantity * supplier["costs"][product_id]
                supplier_quantity += quantity
                total_quantity += quantity
                quality_weighted_sum += supplier["quality"] * quantity

            fixed_cost = supplier["fixed_cost"]
            supplier_costs[supplier_id] = fixed_cost + variable_cost
            supplier_quantities[supplier_id] = supplier_quantity
            total_cost += fixed_cost + variable_cost

        weighted_average_quality = (
            quality_weighted_sum / total_quantity if total_quantity else 0.0
        )
        total_demand = sum(product["demand"] for product in self.data.products.values())
        total_allocated_demand = sum(
            product["demand"] - sum(
                allocation.get(supplier_id, {}).get(product_id, 0)
                for supplier_id in self.data.suppliers
            )
            for product_id, product in self.data.products.items()
        )
        demand_fulfillment = (
            (total_demand - total_allocated_demand) / total_demand
            if total_demand
            else 1.0
        )

        return {
            "total_cost": total_cost,
            "total_quantity": total_quantity,
            "weighted_avg_quality": weighted_average_quality,
            "demand_fulfillment": demand_fulfillment,
            "supplier_costs": supplier_costs,
            "supplier_quantities": supplier_quantities,
            "num_suppliers": len(allocation),
        }

    def solve(self) -> Dict:
        """Run scoring, greedy allocation, and solution measurement in sequence."""
        start_time = time.perf_counter()
        scores, score_details = self.calculate_supplier_scores()
        allocation, unmet_demand, allocation_sequence = self.greedy_allocation(scores)
        metrics = self.calculate_solution_metrics(allocation)
        execution_time = time.perf_counter() - start_time

        # These checks make the model's product-level feasibility rules explicit.
        for product_id, product in self.data.products.items():
            allocated = sum(
                allocation.get(supplier_id, {}).get(product_id, 0)
                for supplier_id in self.data.suppliers
            )
            assert allocated <= product["demand"]
            assert unmet_demand[product_id] == product["demand"] - allocated

        for supplier_id, product_allocations in allocation.items():
            for product_id, quantity in product_allocations.items():
                assert 0 < quantity <= self.data.suppliers[supplier_id]["capacities"][product_id]

        assert np.isclose(
            metrics["total_cost"], sum(metrics["supplier_costs"].values())
        )

        return {
            "allocation": allocation,
            "unmet_demand": unmet_demand,
            "scores": scores,
            "score_details": score_details,
            "allocation_sequence": allocation_sequence,
            "metrics": metrics,
            "execution_time": execution_time,
        }


heuristic = WeightedScoringHeuristic(heuristic_data)
heuristic_solution = heuristic.solve()

print(f"Heuristic run time: {heuristic_solution['execution_time']:.6f} seconds")
print(f"Total procurement cost: ${heuristic_solution['metrics']['total_cost']:,.2f}")
print(f"Selected suppliers: {heuristic_solution['metrics']['num_suppliers']}")
print(
    "Weighted average quality: "
    f"{heuristic_solution['metrics']['weighted_avg_quality']:.1%}"
)
print(f"Demand fulfillment: {heuristic_solution['metrics']['demand_fulfillment']:.1%}")

In [ ]:
def display_heuristic_solution_details(
    solution: Dict, data: SupplierSelectionHeuristicData
) -> None:
    """Print rankings, allocation steps, cost components, and unmet demand."""
    print("=" * 80)
    print("WEIGHTED-SCORING HEURISTIC: SOLUTION DETAILS")
    print("=" * 80)

    print("\n1. SUPPLIER SCORES AND RANKINGS")
    print("-" * 50)
    sorted_scores = sorted(
        solution["scores"].items(), key=lambda item: item[1], reverse=True
    )
    print(f"{'Rank':<5} {'Supplier':<27} {'Score':>8} {'Quality':>9} {'Mean cost':>12} {'Capacity':>12}")
    print("-" * 80)

    for rank, (supplier_id, score) in enumerate(sorted_scores, start=1):
        supplier = data.suppliers[supplier_id]
        details = solution["score_details"][supplier_id]
        print(
            f"{rank:<5} {supplier['name'][:26]:<27} {score:>8.3f} "
            f"{supplier['quality']:>8.1%} ${details['avg_cost']:>10.2f} "
            f"{details['total_capacity']:>11,}"
        )

    print("\n2. GREEDY ALLOCATION SEQUENCE")
    print("-" * 50)
    for step, allocation_step in enumerate(solution["allocation_sequence"], start=1):
        print(
            f"\nStep {step}: {allocation_step['supplier_name']} "
            f"(score {allocation_step['score']:.3f})"
        )
        print(f"  Total allocated: {allocation_step['total_allocated']:,} units")
        for product_id, quantity in allocation_step["allocation_details"].items():
            product_name = data.products[product_id]["name"]
            print(f"  {product_name}: {quantity:,} units")

    metrics = solution["metrics"]
    fixed_costs = sum(
        data.suppliers[supplier_id]["fixed_cost"]
        for supplier_id in solution["allocation"]
    )
    variable_costs = metrics["total_cost"] - fixed_costs

    print("\n3. COST ANALYSIS")
    print("-" * 50)
    print(f"Total procurement cost: ${metrics['total_cost']:,.2f}")
    print(f"  Fixed costs: ${fixed_costs:,.2f}")
    print(f"  Variable costs: ${variable_costs:,.2f}")
    print("Cost by selected supplier:")

    for supplier_id, cost in metrics["supplier_costs"].items():
        supplier = data.suppliers[supplier_id]
        quantity = metrics["supplier_quantities"][supplier_id]
        variable_cost = cost - supplier["fixed_cost"]
        average_unit_cost = variable_cost / quantity if quantity else 0.0
        print(
            f"  {supplier['name']}: ${cost:,.2f} "
            f"({quantity:,} units; variable cost ${average_unit_cost:.2f}/unit)"
        )

    print("\n4. PERFORMANCE MEASURES")
    print("-" * 50)
    print(f"Weighted average quality: {metrics['weighted_avg_quality']:.1%}")
    print(f"Demand fulfillment: {metrics['demand_fulfillment']:.1%}")
    print(f"Number of selected suppliers: {metrics['num_suppliers']}")
    print(f"Measured run time: {solution['execution_time']:.6f} seconds")
    print(
        "Configured supplier-count policy bounds: "
        f"{data.min_suppliers} to {data.max_suppliers}; "
        "these bounds are not enforced by this heuristic."
    )

    unmet_items = [
        (product_id, unmet)
        for product_id, unmet in solution["unmet_demand"].items()
        if unmet > 0
    ]
    print("\n5. UNMET DEMAND")
    print("-" * 50)
    if unmet_items:
        for product_id, unmet in unmet_items:
            demand = data.products[product_id]["demand"]
            name = data.products[product_id]["name"]
            print(f"  {name}: {unmet:,} units ({unmet / demand:.1%} of demand)")
    else:
        print("  No unmet demand.")


display_heuristic_solution_details(heuristic_solution, heuristic_data)

In [ ]:
def create_heuristic_visualizations(
    solution: Dict, data: SupplierSelectionHeuristicData
):
    """Visualize supplier scores, allocation order, costs, and outcome measures."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Weighted-Scoring Heuristic: Solution Analysis", fontsize=16)

    # The radar chart shows normalized criterion scores, not raw supplier measurements.
    ax1 = axes[0, 0]
    categories = ["Cost", "Quality", "Reliability", "Capacity", "Sustainability"]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    top_suppliers = sorted(
        solution["scores"].items(), key=lambda item: item[1], reverse=True
    )[:3]
    colors = plt.cm.Set1(np.linspace(0, 1, max(len(top_suppliers), 1)))

    for color, (supplier_id, _) in zip(colors, top_suppliers):
        criterion_scores = solution["score_details"][supplier_id]["criterion_scores"]
        values = [
            criterion_scores[criterion] * 100
            for criterion in (
                "cost", "quality", "reliability", "capacity", "sustainability"
            )
        ]
        ax1.plot(
            closed_angles,
            values + values[:1],
            marker="o",
            linewidth=2,
            label=data.suppliers[supplier_id]["name"],
            color=color,
        )
        ax1.fill(closed_angles, values + values[:1], alpha=0.12, color=color)

    ax1.set_xticks(angles)
    ax1.set_xticklabels(categories)
    ax1.set_ylim(0, 100)
    ax1.set_ylabel("Normalized criterion score (%)")
    ax1.set_title("Highest-Ranked Suppliers: Criterion Scores")
    ax1.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12))
    ax1.grid(True)

    # Bar colors are mapped to the displayed supplier scores and explained by a colorbar.
    ax2 = axes[0, 1]
    sequence = solution["allocation_sequence"]
    names = [step["supplier_name"] for step in sequence]
    scores = np.array([step["score"] for step in sequence], dtype=float)
    quantities = [step["total_allocated"] for step in sequence]
    positions = np.arange(len(names))

    if names:
        score_min = float(scores.min())
        score_max = float(scores.max())
        if np.isclose(score_min, score_max):
            score_max = score_min + 1e-12
        color_map = plt.cm.RdYlGn
        normalization = plt.Normalize(vmin=score_min, vmax=score_max)
        bars = ax2.bar(positions, quantities, color=color_map(normalization(scores)))
        colorbar = fig.colorbar(
            plt.cm.ScalarMappable(norm=normalization, cmap=color_map), ax=ax2
        )
        colorbar.set_label("Composite supplier score")
        for position, quantity, score in zip(positions, quantities, scores):
            ax2.text(position, quantity, f"{score:.3f}", ha="center", va="bottom")
    else:
        bars = ax2.bar([], [])

    ax2.set_xticks(positions)
    ax2.set_xticklabels(names, rotation=35, ha="right")
    ax2.set_xlabel("Suppliers in allocation order")
    ax2.set_ylabel("Allocated quantity (units)")
    ax2.set_title("Greedy Allocation Sequence")
    ax2.grid(True, axis="y", alpha=0.3)

    ax3 = axes[1, 0]
    selected_ids = list(solution["allocation"])
    selected_names = [data.suppliers[supplier_id]["name"] for supplier_id in selected_ids]
    fixed_costs = [data.suppliers[supplier_id]["fixed_cost"] for supplier_id in selected_ids]
    variable_costs = [
        solution["metrics"]["supplier_costs"][supplier_id]
        - data.suppliers[supplier_id]["fixed_cost"]
        for supplier_id in selected_ids
    ]
    cost_positions = np.arange(len(selected_ids))
    ax3.bar(cost_positions, fixed_costs, label="Fixed cost")
    ax3.bar(
        cost_positions,
        variable_costs,
        bottom=fixed_costs,
        label="Variable cost",
    )
    ax3.set_xticks(cost_positions)
    ax3.set_xticklabels(selected_names, rotation=35, ha="right")
    ax3.set_ylabel("Cost (currency units)")
    ax3.set_title("Cost Components by Selected Supplier")
    ax3.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda value, _: f"${value:,.0f}")
    )
    ax3.legend()
    ax3.grid(True, axis="y", alpha=0.3)

    ax4 = axes[1, 1]
    metrics = solution["metrics"]
    metric_names = ["Weighted quality", "Demand fulfillment", "Supplier share"]
    metric_values = [
        metrics["weighted_avg_quality"] * 100,
        metrics["demand_fulfillment"] * 100,
        (metrics["num_suppliers"] / len(data.suppliers) * 100)
        if data.suppliers
        else 0.0,
    ]
    metric_positions = np.arange(len(metric_names))
    metric_bars = ax4.bar(metric_positions, metric_values, color="steelblue")
    ax4.set_xticks(metric_positions)
    ax4.set_xticklabels(metric_names, rotation=15, ha="right")
    ax4.set_ylabel("Percentage (%)")
    ax4.set_ylim(0, 100)
    ax4.set_title("Outcome Measures")
    ax4.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(metric_bars, metric_values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}%",
            ha="center",
            va="bottom",
        )

    fig.tight_layout()
    plt.show()
    return fig


heuristic_fig = create_heuristic_visualizations(heuristic_solution, heuristic_data)

In [ ]:
def compare_with_optimal(
    heuristic_solution: Dict, data: SupplierSelectionHeuristicData
) -> Dict:
    """Compare heuristic cost with a transparent, valid cost lower bound."""
    print("=" * 80)
    print("HEURISTIC COST AND LOWER-BOUND COMPARISON")
    print("=" * 80)
    print("\n1. COST LOWER BOUND")
    print("-" * 50)

    # Independent product-level minimum costs relax capacity and supplier coupling.
    minimum_variable_cost = sum(
        min(data.suppliers[supplier_id]["costs"][product_id]
            for supplier_id in data.suppliers)
        * product["demand"]
        for product_id, product in data.products.items()
    )
    sorted_fixed_costs = sorted(
        supplier["fixed_cost"] for supplier in data.suppliers.values()
    )
    required_supplier_count = min(data.min_suppliers, len(sorted_fixed_costs))
    minimum_fixed_cost = sum(sorted_fixed_costs[:required_supplier_count])
    cost_lower_bound = minimum_variable_cost + minimum_fixed_cost

    print(f"Independent minimum variable-cost bound: ${minimum_variable_cost:,.2f}")
    print(f"Minimum fixed-cost bound: ${minimum_fixed_cost:,.2f}")
    print(f"Combined cost lower bound: ${cost_lower_bound:,.2f}")
    print(
        "This is a lower bound, not a proven optimal solution: it relaxes "
        "capacity and combines independently favorable choices."
    )

    heuristic_cost = heuristic_solution["metrics"]["total_cost"]
    absolute_bound_gap = heuristic_cost - cost_lower_bound
    relative_bound_gap = (
        absolute_bound_gap / cost_lower_bound * 100 if cost_lower_bound else 0.0
    )

    print("\n2. HEURISTIC COST RELATIVE TO THE LOWER BOUND")
    print("-" * 50)
    print(f"Heuristic procurement cost: ${heuristic_cost:,.2f}")
    print(f"Difference from lower bound: ${absolute_bound_gap:,.2f}")
    print(f"Difference as a percentage of the bound: {relative_bound_gap:.2f}%")
    print(
        "The percentage is a bound gap, not an optimality gap; the true optimum "
        "is not calculated in this notebook."
    )

    print("\n3. COMPUTATIONAL EFFORT")
    print("-" * 50)
    supplier_count = len(data.suppliers)
    product_count = len(data.products)
    criterion_count = len(data.weights)
    print(f"Suppliers: {supplier_count}")
    print(f"Products: {product_count}")
    print(f"Supplier-product pairs: {supplier_count * product_count}")
    print(f"Scoring criteria: {criterion_count}")
    print(f"Measured heuristic run time: {heuristic_solution['execution_time']:.6f} seconds")
    print(
        "The measured time is specific to this run and environment. The algorithmic "
        "time complexity is O(|I||K| + |I| log |I| + |I||J|)."
    )

    return {
        "cost_lower_bound": cost_lower_bound,
        "heuristic_cost": heuristic_cost,
        "absolute_bound_gap": absolute_bound_gap,
        "relative_bound_gap_percent": relative_bound_gap,
        "execution_time": heuristic_solution["execution_time"],
    }


comparison_results = compare_with_optimal(heuristic_solution, heuristic_data)

In [ ]:
def heuristic_sensitivity_analysis(
    data: SupplierSelectionHeuristicData,
) -> Dict:
    """Examine the effects of weights, policy bounds, and demand changes."""
    print("=" * 80)
    print("HEURISTIC SENSITIVITY ANALYSIS")
    print("=" * 80)
    sensitivity_results = {}

    print("\n1. WEIGHT VARIATION")
    print("-" * 40)
    weight_scenarios = {
        "Cost-focused": {
            "cost": 0.50, "quality": 0.20, "reliability": 0.15,
            "capacity": 0.10, "sustainability": 0.05,
        },
        "Quality-focused": {
            "cost": 0.15, "quality": 0.50, "reliability": 0.20,
            "capacity": 0.10, "sustainability": 0.05,
        },
        "Balanced": {
            "cost": 0.30, "quality": 0.25, "reliability": 0.20,
            "capacity": 0.15, "sustainability": 0.10,
        },
        "Sustainability-focused": {
            "cost": 0.20, "quality": 0.20, "reliability": 0.15,
            "capacity": 0.10, "sustainability": 0.35,
        },
    }

    for scenario_name, scenario_weights in weight_scenarios.items():
        assert np.isclose(sum(scenario_weights.values()), 1.0)
        modified_data = SupplierSelectionHeuristicData()
        modified_data.weights = scenario_weights.copy()
        modified_solution = WeightedScoringHeuristic(modified_data).solve()
        metrics = modified_solution["metrics"]

        print(f"\n{scenario_name} weights:")
        print(f"  Procurement cost: ${metrics['total_cost']:,.2f}")
        print(f"  Weighted average quality: {metrics['weighted_avg_quality']:.1%}")
        print(f"  Selected suppliers: {metrics['num_suppliers']}")

        sensitivity_results[scenario_name] = {
            "total_cost": metrics["total_cost"],
            "quality": metrics["weighted_avg_quality"],
            "suppliers": metrics["num_suppliers"],
            "weights": scenario_weights.copy(),
        }

    print("\n2. SUPPLIER-COUNT POLICY-BOUND DIAGNOSTIC")
    print("-" * 40)
    print(
        "The configured minimum and maximum supplier counts do not change the "
        "greedy allocation; this section checks whether its result falls within each range."
    )
    supplier_constraints = [(1, 2), (2, 3), (3, 4), (2, 5)]

    for minimum, maximum in supplier_constraints:
        modified_data = SupplierSelectionHeuristicData()
        modified_data.min_suppliers = minimum
        modified_data.max_suppliers = maximum
        modified_solution = WeightedScoringHeuristic(modified_data).solve()
        selected_count = modified_solution["metrics"]["num_suppliers"]
        satisfies_bounds = minimum <= selected_count <= maximum
        status = "within bounds" if satisfies_bounds else "outside bounds"
        print(
            f"  Range {minimum}-{maximum}: {selected_count} selected; "
            f"{status}. The bounds are diagnostic only."
        )

    print("\n3. DEMAND VARIATION")
    print("-" * 40)
    demand_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
    demand_results = {}

    for multiplier in demand_multipliers:
        modified_data = SupplierSelectionHeuristicData()
        for product_id in modified_data.products:
            modified_data.products[product_id]["demand"] = int(
                data.products[product_id]["demand"] * multiplier
            )
        modified_solution = WeightedScoringHeuristic(modified_data).solve()
        metrics = modified_solution["metrics"]
        print(
            f"  Demand multiplier {multiplier:.1f}: "
            f"cost ${metrics['total_cost']:,.2f}, "
            f"{metrics['num_suppliers']} suppliers, "
            f"{metrics['demand_fulfillment']:.1%} fulfillment"
        )
        demand_results[multiplier] = {
            "total_cost": metrics["total_cost"],
            "suppliers": metrics["num_suppliers"],
            "demand_fulfillment": metrics["demand_fulfillment"],
        }

    sensitivity_results["demand_variation"] = demand_results
    return sensitivity_results


sensitivity_results = heuristic_sensitivity_analysis(heuristic_data)

In [ ]:
def create_sensitivity_visualization(sensitivity_results: Dict):
    """Plot weight-scenario costs and quality without implying an optimum."""
    scenario_names = [
        name for name in sensitivity_results if name != "demand_variation"
    ]
    if not scenario_names:
        raise ValueError("At least one weight scenario is required.")

    costs = [sensitivity_results[name]["total_cost"] for name in scenario_names]
    qualities = [sensitivity_results[name]["quality"] * 100 for name in scenario_names]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("Sensitivity to Weighted-Scoring Criteria", fontsize=15)

    ax1 = axes[0]
    positions = np.arange(len(scenario_names))
    bars = ax1.bar(positions, costs, color="steelblue", label="Procurement cost")
    ax1.set_xticks(positions)
    ax1.set_xticklabels(scenario_names, rotation=30, ha="right")
    ax1.set_ylabel("Procurement cost (currency units)")
    ax1.set_title("Cost by Weight Scenario")
    ax1.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda value, _: f"${value / 1000:,.0f}K")
    )
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, cost in zip(bars, costs):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${cost:,.0f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax2 = axes[1]
    minimum_cost = min(costs)
    maximum_cost = max(costs)
    if np.isclose(minimum_cost, maximum_cost):
        normalized_costs = np.zeros(len(costs))
    else:
        normalized_costs = [
            (cost - minimum_cost) / (maximum_cost - minimum_cost) * 100
            for cost in costs
        ]

    ax2.scatter(normalized_costs, qualities, color="darkred", s=70)
    for cost_position, quality, name in zip(
        normalized_costs, qualities, scenario_names
    ):
        ax2.annotate(
            name,
            (cost_position, quality),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )
    ax2.set_xlabel("Normalized scenario cost (lower is less costly)")
    ax2.set_ylabel("Weighted average quality (%)")
    ax2.set_title("Cost and Quality Across Weight Scenarios")
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


sensitivity_fig = create_sensitivity_visualization(sensitivity_results)

## Summary and interpretation

The notebook implements a weighted-scoring heuristic in a fixed sequence: normalize supplier-level criteria, combine them using decision-maker weights, rank suppliers, allocate products greedily in that ranking order, and measure the resulting plan. Product-specific demand and capacity determine feasible quantities. Fixed costs are charged once for each supplier that receives a positive allocation; variable costs are charged for every allocated unit.

Use the printed tables and plots to trace the method rather than treating the composite score as a procurement-cost measure. A high score reflects the chosen criteria and weights; it does not prove that the supplier is the least-cost choice for every product or that the resulting allocation is optimal. The lower-bound comparison is informative as a reference, but its percentage difference is not an optimality gap because this notebook does not calculate a mathematical optimum.

The weight sensitivity experiment shows how changing decision priorities can change rankings and outcomes. The demand experiment shows how the same ranked allocation rule responds to changing quantities and available capacity. The supplier-count diagnostic reports whether a result lies within configured policy bounds; those bounds do not constrain this heuristic's allocation.

### Strengths and limitations

- **Interpretability:** The score is a weighted sum of normalized criteria, and the allocation sequence can be inspected step by step.
- **Computational effort:** Scoring, sorting, and allocation have polynomial running time, with complexity stated in terms of suppliers, products, and criteria. The measured execution time is specific to the running environment and should not be treated as a general performance guarantee.
- **Weight dependence:** Rankings depend on the chosen criteria, normalization ranges, and weights. Sensitivity analysis helps reveal this dependence but does not determine the uniquely correct weights.
- **Greedy limitation:** A locally preferred supplier is processed first even when another sequence could produce a lower total cost or a different quality-cost trade-off. No optimality guarantee is provided.
- **Scope of constraints:** Capacity and demand are handled product by product. Minimum and maximum supplier counts are reported but not enforced, and other operational considerations are outside this example's model.

### Further study

Natural extensions include representing additional operational requirements as explicit constraints and testing the same scoring-and-greedy procedure on larger, appropriately specified instances. Such extensions require their assumptions and effects on feasibility to be stated clearly.